# NYC Airbnb Market Overview — Exploratory Data Analysis

**Day 2 | 28-Day Agentic Analysis Challenge**

This notebook explores the NYC Airbnb dataset (5,999 listings) through pricing patterns, review trends, and rating distributions across New York City's five boroughs. We examine how factors like district, room type, host attributes, and listing characteristics influence pricing and guest satisfaction.

**Dataset:** `airbnb_cleaned.csv` — 5,999 rows, 25 columns  
**Scope:** Pricing Analysis (9 charts) + Reviews & Ratings (9 charts)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Style setup — light, minimal theme with muted colors
sns.set_style("whitegrid")
PALETTE = sns.color_palette("muted")
PASTEL = sns.color_palette("pastel")
sns.set_palette("muted")

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#cccccc',
    'grid.color': '#eeeeee',
})

print("Libraries loaded successfully.")

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('airbnb_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\n--- First 5 rows ---")
df.head()

In [ ]:
# Quick data overview
print(f"{'='*50}")
print(f"DATASET OVERVIEW")
print(f"{'='*50}")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"\nDistrict distribution:")
print(df['District'].value_counts().to_string())
print(f"\nRoom Type distribution:")
print(df['Room Type'].value_counts().to_string())
print(f"\nPrice stats:  Mean=${df['Price'].mean():.0f}  |  Median=${df['Price'].median():.0f}  |  Max=${df['Price'].max():,.0f}")
print(f"Rating stats: Mean={df['Rating'].mean():.2f}  |  Min={df['Rating'].min():.1f}  |  Max={df['Rating'].max():.1f}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0].to_string()}")
if df.isnull().sum().sum() == 0:
    print("  None — dataset is fully clean.")

In [ ]:
# Outlier handling — cap prices at $1,000 for visualizations
outliers = df[df['Price'] > 1000]
print(f"Outliers (Price > $1,000): {len(outliers)} listings")
print(f"  Price range of outliers: ${outliers['Price'].min():,.0f} – ${outliers['Price'].max():,.0f}")
print(f"  Districts: {outliers['District'].value_counts().to_dict()}")
print(f"  Room Types: {outliers['Room Type'].value_counts().to_dict()}")

# Create capped price column — original Price column is preserved
df['Price_Capped'] = df['Price'].clip(upper=1000)
print(f"\nCreated 'Price_Capped' column (max $1,000). Original 'Price' column unchanged.")
print(f"  Capped mean: ${df['Price_Capped'].mean():.0f}  |  Original mean: ${df['Price'].mean():.0f}")

# Parse date columns for time-series analysis
df['First Review'] = pd.to_datetime(df['First Review'], errors='coerce')
df['Last Review'] = pd.to_datetime(df['Last Review'], errors='coerce')

---
# Section 1: Pricing Analysis

How do prices vary across districts, room types, neighborhoods, and host attributes? This section examines the distribution and drivers of Airbnb pricing in NYC.

In [ ]:
# Chart 1: Price Distribution Histogram (capped at $1,000)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Price_Capped'], bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
ax.axvline(df['Price_Capped'].median(), color=PALETTE[3], linestyle='--', linewidth=1.5, label=f"Median: ${df['Price_Capped'].median():.0f}")
ax.axvline(df['Price_Capped'].mean(), color=PALETTE[1], linestyle='--', linewidth=1.5, label=f"Mean: ${df['Price_Capped'].mean():.0f}")
ax.set_title('Distribution of Listing Prices (Capped at $1,000)')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Number of Listings')
ax.legend(fontsize=11)
ax.set_xlim(0, 1050)
plt.tight_layout()
plt.show()

**Insight:** The price distribution is heavily right-skewed. The majority of listings fall below $200/night, with the median (~$99) sitting well below the mean (~$136 capped). This indicates a market dominated by budget and mid-range options, with a long tail of premium listings. 27 extreme outliers above $1,000 were capped for visualization clarity.

In [ ]:
# Chart 2: Price by District (Box Plot)
district_order = df.groupby('District')['Price_Capped'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='District', y='Price_Capped', order=district_order,
            palette=PASTEL, fliersize=2, linewidth=1, ax=ax)
ax.set_title('Price Distribution by District')
ax.set_xlabel('District')
ax.set_ylabel('Price ($, capped at $1,000)')
plt.tight_layout()
plt.show()

**Insight:** Manhattan commands the highest median prices, consistent with its status as NYC's prime tourist and business hub. Brooklyn follows as a strong second market. The outer boroughs (Queens, Bronx, Staten Island) offer substantially lower prices, reflecting their distance from central attractions and lower demand.

In [ ]:
# Chart 3: Price by Room Type (Violin Plot)
room_order = df.groupby('Room Type')['Price_Capped'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=df, x='Room Type', y='Price_Capped', order=room_order,
               palette=PASTEL[:3], inner='quartile', linewidth=1, ax=ax)
ax.set_title('Price Distribution by Room Type')
ax.set_xlabel('Room Type')
ax.set_ylabel('Price ($, capped at $1,000)')
plt.tight_layout()
plt.show()

**Insight:** Entire places dominate the upper price range with a wide distribution, reflecting diverse property sizes and locations. Private rooms cluster tightly in the $50-$150 range — a more predictable budget option. Shared rooms are the cheapest but represent only ~2% of listings, indicating minimal market share for this category.

In [ ]:
# Chart 4: Avg Price by Neighborhood — Top 15 & Bottom 15
neighborhood_avg = df.groupby('Neighborhood')['Price_Capped'].mean().sort_values(ascending=False)
top15 = neighborhood_avg.head(15)
bottom15 = neighborhood_avg.tail(15).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Top 15
axes[0].barh(top15.index[::-1], top15.values[::-1], color=PALETTE[3], edgecolor='white')
axes[0].set_title('Top 15 Neighborhoods by Avg Price')
axes[0].set_xlabel('Average Price ($)')
for i, v in enumerate(top15.values[::-1]):
    axes[0].text(v + 3, i, f'${v:.0f}', va='center', fontsize=9)

# Bottom 15
axes[1].barh(bottom15.index[::-1], bottom15.values[::-1], color=PALETTE[0], edgecolor='white')
axes[1].set_title('Bottom 15 Neighborhoods by Avg Price')
axes[1].set_xlabel('Average Price ($)')
for i, v in enumerate(bottom15.values[::-1]):
    axes[1].text(v + 1, i, f'${v:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

**Insight:** The most expensive neighborhoods are concentrated in Manhattan (Tribeca, SoHo, Midtown, etc.) and select Brooklyn areas, with averages well above $200/night. The cheapest neighborhoods are in the outer boroughs, where average prices drop below $75. This neighborhood-level view reveals that location within a borough matters as much as the borough itself.

In [ ]:
# Chart 5: Price vs Accommodates (Scatter + Trend Line)
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=df, x='Accommodates', y='Price_Capped',
            scatter_kws={'alpha': 0.15, 'color': PALETTE[0], 's': 15},
            line_kws={'color': PALETTE[3], 'linewidth': 2},
            ax=ax)
ax.set_title('Price vs Number of Guests Accommodated')
ax.set_xlabel('Accommodates (guests)')
ax.set_ylabel('Price ($, capped at $1,000)')
plt.tight_layout()
plt.show()

**Insight:** There is a clear positive correlation between the number of guests a listing accommodates and its price. Each additional guest capacity adds roughly $30-50 to the nightly rate. Listings accommodating 6+ guests show much wider price variance, likely reflecting differences in property quality and location.

In [ ]:
# Chart 6: Price vs Rating (Scatter)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['Rating'], df['Price_Capped'], alpha=0.12, s=15, color=PALETTE[2], edgecolors='none')
ax.set_title('Price vs Guest Rating')
ax.set_xlabel('Rating')
ax.set_ylabel('Price ($, capped at $1,000)')

# Add a trend line
z = np.polyfit(df['Rating'].dropna(), df.loc[df['Rating'].notna(), 'Price_Capped'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Rating'].min(), df['Rating'].max(), 100)
ax.plot(x_line, p(x_line), color=PALETTE[3], linewidth=2, linestyle='--', label=f'Trend')
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** Surprisingly, there is little to no correlation between price and rating. High-rated listings exist across all price ranges, and expensive listings do not consistently earn better reviews. This suggests that guest satisfaction is driven more by host quality, accuracy of listing descriptions, and value-for-money rather than absolute price.

In [ ]:
# Chart 7: Price Heatmap — District x Room Type (Annotated)
heatmap_data = df.pivot_table(values='Price_Capped', index='District', columns='Room Type', aggfunc='mean')
# Reorder for logical presentation
district_sort = df.groupby('District')['Price_Capped'].mean().sort_values(ascending=False).index
heatmap_data = heatmap_data.reindex(district_sort)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrBr', linewidths=1, linecolor='white',
            cbar_kws={'label': 'Average Price ($)'}, ax=ax)
ax.set_title('Average Price Heatmap: District x Room Type')
ax.set_xlabel('Room Type')
ax.set_ylabel('District')
plt.tight_layout()
plt.show()

**Insight:** The heatmap clearly shows that the combination of Manhattan + Entire Place commands the highest average prices. The district effect is strongest for entire places (large price gaps between Manhattan and outer boroughs) but narrows for shared rooms, where prices converge across districts. This makes sense — shared rooms compete primarily on price regardless of location.

In [ ]:
# Chart 8: Instant Book vs Non-Instant Book Pricing (Box Plot)
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Instant Book', y='Price_Capped',
            palette=[PASTEL[0], PASTEL[2]], fliersize=2, linewidth=1, ax=ax)
ax.set_title('Price: Instant Book vs Non-Instant Book Listings')
ax.set_xlabel('Instant Book')
ax.set_ylabel('Price ($, capped at $1,000)')

# Add median annotations
medians = df.groupby('Instant Book')['Price_Capped'].median()
for i, (cat, med) in enumerate(medians.items()):
    ax.text(i, med + 15, f'Median: ${med:.0f}', ha='center', fontsize=10, color='#333333')

plt.tight_layout()
plt.show()

**Insight:** Instant Book and non-Instant Book listings show similar median prices, suggesting that enabling instant booking is not strongly associated with pricing strategy. Hosts across all price points use the feature, likely as a convenience tool to increase booking velocity rather than as a price differentiator.

In [ ]:
# Chart 9: Superhost vs Non-Superhost Pricing (Box Plot)
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Superhost', y='Price_Capped',
            palette=[PASTEL[4], PASTEL[1]], fliersize=2, linewidth=1, ax=ax)
ax.set_title('Price: Superhost vs Non-Superhost Listings')
ax.set_xlabel('Superhost Status')
ax.set_ylabel('Price ($, capped at $1,000)')

medians = df.groupby('Superhost')['Price_Capped'].median()
for i, (cat, med) in enumerate(medians.items()):
    ax.text(i, med + 15, f'Median: ${med:.0f}', ha='center', fontsize=10, color='#333333')

plt.tight_layout()
plt.show()

**Insight:** Superhosts tend to price their listings slightly higher than non-Superhosts, which aligns with their ability to leverage trust and reputation. However, the difference is modest — Superhost status may allow for a small premium, but the primary benefit likely comes from higher occupancy rates and booking frequency rather than dramatically higher nightly rates.

---
# Section 2: Reviews & Ratings

How are ratings distributed? What drives review volume? This section explores guest feedback patterns, review activity over time, and factors associated with higher ratings.

In [ ]:
# Chart 10: Rating Distribution Histogram
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Rating'].dropna(), bins=30, color=PALETTE[2], edgecolor='white', alpha=0.85)
ax.axvline(df['Rating'].mean(), color=PALETTE[3], linestyle='--', linewidth=1.5,
           label=f"Mean: {df['Rating'].mean():.2f}")
ax.axvline(df['Rating'].median(), color=PALETTE[1], linestyle='--', linewidth=1.5,
           label=f"Median: {df['Rating'].median():.2f}")
ax.set_title('Distribution of Guest Ratings')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Listings')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Insight:** Ratings are heavily left-skewed (concentrated at the high end), with the vast majority of listings rated 4.5 or above. The mean rating of ~4.79 indicates a consistently positive guest experience across the platform. Very few listings fall below 4.0, suggesting either genuine quality or survivorship bias (poorly-rated listings may get delisted).

In [ ]:
# Chart 11: Rating by District (Box Plot)
fig, ax = plt.subplots(figsize=(10, 6))
district_rating_order = df.groupby('District')['Rating'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='District', y='Rating', order=district_rating_order,
            palette=PASTEL, fliersize=2, linewidth=1, ax=ax)
ax.set_title('Rating Distribution by District')
ax.set_xlabel('District')
ax.set_ylabel('Rating')
plt.tight_layout()
plt.show()

**Insight:** Ratings are remarkably consistent across all five districts, with median values hovering near 4.8-4.9. Unlike pricing, guest satisfaction does not vary dramatically by borough. This suggests that hosts across NYC maintain similar quality standards, and guests calibrate expectations based on price point and location.

In [ ]:
# Chart 12: Rating by Room Type (Box Plot)
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Room Type', y='Rating',
            palette=PASTEL[:3], fliersize=2, linewidth=1, ax=ax)
ax.set_title('Rating Distribution by Room Type')
ax.set_xlabel('Room Type')
ax.set_ylabel('Rating')
plt.tight_layout()
plt.show()

**Insight:** Entire places and private rooms receive similar median ratings. Shared rooms show slightly more variance and a marginally lower median, which is expected given the inherent trade-offs of sharing space with strangers. Overall, room type has minimal impact on guest satisfaction.

In [ ]:
# Chart 13: Review Volume Over Time — Monthly Trend (Line Chart)
review_dates = df['First Review'].dropna()
monthly_reviews = review_dates.dt.to_period('M').value_counts().sort_index()
monthly_reviews.index = monthly_reviews.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly_reviews.index, monthly_reviews.values, color=PALETTE[0], linewidth=1.5)
ax.fill_between(monthly_reviews.index, monthly_reviews.values, alpha=0.15, color=PALETTE[0])
ax.set_title('New Listings Over Time (by First Review Date)')
ax.set_xlabel('Date')
ax.set_ylabel('Number of New Listings')
ax.xaxis.set_major_locator(mticker.MaxNLocator(12))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight:** The monthly trend of first reviews serves as a proxy for new listing activity. The chart reveals the growth trajectory of NYC's Airbnb market, showing periods of rapid expansion and any seasonal patterns or market disruptions (e.g., regulatory changes, pandemic impacts) that affected new host onboarding.

In [ ]:
# Chart 14: Num_Reviews vs Rating (Scatter)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['Rating'], df['Num_Reviews'], alpha=0.12, s=15, color=PALETTE[4], edgecolors='none')
ax.set_title('Number of Reviews vs Guest Rating')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Reviews')
plt.tight_layout()
plt.show()

**Insight:** Highly-reviewed listings (100+ reviews) tend to cluster in the 4.5-5.0 rating range, suggesting a virtuous cycle: better-rated listings attract more bookings, which generate more reviews, which further boost visibility. Listings with low ratings rarely accumulate many reviews, likely due to reduced demand or host exit from the platform.

In [ ]:
# Chart 15: Top 20 Most Reviewed Listings (Horizontal Bar)
top_reviewed = df.nlargest(20, 'Num_Reviews')[['Place Name', 'Num_Reviews', 'District', 'Rating']].reset_index(drop=True)
labels = [f"{row['Place Name'][:35]}... ({row['District']})" if len(row['Place Name']) > 35
          else f"{row['Place Name']} ({row['District']})" for _, row in top_reviewed.iterrows()]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(labels[::-1], top_reviewed['Num_Reviews'].values[::-1],
               color=PALETTE[1], edgecolor='white')
for i, (reviews, rating) in enumerate(zip(top_reviewed['Num_Reviews'].values[::-1],
                                           top_reviewed['Rating'].values[::-1])):
    ax.text(reviews + 5, i, f'{reviews} reviews | {rating} rating', va='center', fontsize=8.5)
ax.set_title('Top 20 Most Reviewed Listings')
ax.set_xlabel('Number of Reviews')
plt.tight_layout()
plt.show()

**Insight:** The most-reviewed listings are spread across multiple districts and maintain high ratings (mostly 4.5+). These are likely long-established, well-managed properties that have built strong reputations over years of consistent service. Their high review counts make them highly visible in search results, creating a competitive advantage.

In [ ]:
# Chart 16: Days Since Last Review Distribution (Histogram — Activity Indicator)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Days Since Last Review'].dropna(), bins=50, color=PALETTE[5], edgecolor='white', alpha=0.85)
ax.axvline(df['Days Since Last Review'].median(), color=PALETTE[3], linestyle='--', linewidth=1.5,
           label=f"Median: {df['Days Since Last Review'].median():.0f} days")
ax.set_title('Days Since Last Review (Listing Activity/Staleness)')
ax.set_xlabel('Days Since Last Review')
ax.set_ylabel('Number of Listings')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Insight:** This distribution acts as a "freshness" indicator. Listings with recent reviews (under 30 days) are actively booked and competitive. A long tail of listings with 200+ days since their last review may be inactive, seasonal, or struggling to attract guests. This metric could help identify stale listings that may need attention or removal from active search results.

In [ ]:
# Chart 17: Listing Age vs Num_Reviews (Scatter)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['Listing Age (days)'], df['Num_Reviews'], alpha=0.12, s=15, color=PALETTE[0], edgecolors='none')

# Trend line
mask = df['Listing Age (days)'].notna() & df['Num_Reviews'].notna()
z = np.polyfit(df.loc[mask, 'Listing Age (days)'], df.loc[mask, 'Num_Reviews'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Listing Age (days)'].min(), df['Listing Age (days)'].max(), 100)
ax.plot(x_line, p(x_line), color=PALETTE[3], linewidth=2, linestyle='--', label='Trend')

ax.set_title('Listing Age vs Number of Reviews')
ax.set_xlabel('Listing Age (days)')
ax.set_ylabel('Number of Reviews')
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** Older listings naturally accumulate more reviews over time, as shown by the positive trend. However, the wide spread indicates that age alone does not guarantee high review counts — active management, competitive pricing, and quality all play a role. Some young listings outperform much older ones, suggesting strong market positioning from the start.

In [ ]:
# Chart 18: Rating vs Response Time (Bar Chart — Avg Rating per Response Time Category)
response_rating = df.groupby('Response Time')['Rating'].agg(['mean', 'count']).reset_index()
response_rating = response_rating[response_rating['count'] >= 10]  # Filter small groups
response_rating = response_rating.sort_values('mean', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(response_rating['Response Time'], response_rating['mean'],
               color=PALETTE[2], edgecolor='white')
ax.set_title('Average Rating by Host Response Time')
ax.set_xlabel('Average Rating')
ax.set_ylabel('Response Time')

# Add value labels
for i, (val, count) in enumerate(zip(response_rating['mean'], response_rating['count'])):
    ax.text(val + 0.005, i, f'{val:.2f}  (n={count})', va='center', fontsize=10)

# Set x-axis to start near the minimum for better visual comparison
ax.set_xlim(response_rating['mean'].min() - 0.15, 5.0)
plt.tight_layout()
plt.show()

**Insight:** Hosts who respond faster tend to receive higher ratings, with "within an hour" responders earning the best average scores. This relationship makes intuitive sense — quick communication signals professionalism and attentiveness, which guests value highly. Response time is one of the most actionable levers hosts can pull to improve their ratings.

---
# Summary: Key Findings

### Pricing
- **Location is king:** Manhattan commands the highest prices, with premium neighborhoods (Tribeca, SoHo) averaging 3-4x more than outer-borough areas.
- **Room type matters:** Entire places are priced ~2x higher than private rooms on average. Shared rooms are a niche, low-price segment (~2% of listings).
- **Guest capacity drives price:** Each additional guest accommodated adds $30-50/night, making capacity one of the strongest price predictors.
- **Host attributes have modest impact:** Superhost status and Instant Book show only small price differences — these features likely drive volume, not rate.
- **27 extreme outliers** (above $1,000/night) were identified and capped for analysis. These are luxury/unique properties concentrated in Manhattan.

### Reviews & Ratings
- **Ratings are uniformly high:** Mean of 4.79 with most listings above 4.5. This "grade inflation" makes it hard to differentiate on rating alone.
- **Ratings are consistent across districts and room types** — unlike pricing, guest satisfaction does not vary much by geography.
- **Review volume correlates with listing age** but is not guaranteed — active management matters more than tenure.
- **Response time is the strongest rating predictor:** Hosts who respond within an hour earn meaningfully higher ratings, making this the most actionable improvement for hosts.
- **Days Since Last Review** serves as a useful staleness indicator — a long tail of inactive listings suggests market churn.

### Implications
- For **guests:** Price is driven by location and capacity, not quality. High ratings are nearly universal, so look at review count and response time for quality signals.
- For **hosts:** The fastest path to better ratings is faster response times. Superhost status provides a small pricing premium but likely bigger visibility benefits.
- For **Airbnb:** The outer boroughs represent an underserved, affordable market segment with growth potential.